# Phase 0 — Scaffolding

Five packages in a `uv` workspace: `jt_core` (shared math), `tycho` (simulator), `jan`
(strategies), `rudolf` (referee + all plotting) and `jt_bench` (harness, sweeps, `jt` CLI).
This notebook demonstrates the Phase 0 deliverables: the import boundary, seeded random
sub-streams, and the visualization infrastructure.

In [1]:
import os, warnings
os.chdir('/home/dylan/Projects/jan-and-tycho')
os.environ.setdefault("MPLBACKEND", "Agg")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from pathlib import Path
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 160)

## The information boundary

`import-linter` contracts forbid Jan from importing Tycho or Rudolf. A test also checks that a deliberate violation breaks the build (`tests/test_boundary.py`).

In [2]:
import subprocess
print(subprocess.run(["lint-imports", "--no-cache"], capture_output=True, text=True).stdout[-600:])

━━━━━━━━━━━━━━━━━ 4/4
Checking rudolf may import jt_core only
🧐 Checking contracts ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4

---------
Contracts
---------

Analyzed 86 files, 309 dependencies.
------------------------------------

jt_core imports nothing from the other packages KEPT
tycho may import jt_core only KEPT
jan may import jt_core only (never tycho or rudolf) KEPT
rudolf may import jt_core only KEPT

Contracts: 4 kept, 0 broken.



## Reproducible randomness

Every random component draws from a named sub-stream of the scenario seed, so changing one component never reshuffles another.

In [3]:
from jt_core import rng
a = rng.generator(7, "noise").normal(size=3)
b = rng.generator(7, "noise").normal(size=3)
c = rng.generator(7, "schedule").normal(size=3)
print("same stream twice:", np.array_equal(a, b), "| different stream:", np.allclose(a, c))

same stream twice: True | different stream: False


## Visualization infrastructure

A stable strategy → colour map, the figure manifest, and an animation writer that uses ffmpeg (MP4) when present and falls back to GIF.

In [4]:
import matplotlib.pyplot as plt
from rudolf.viz import style, writers
names = ["ptolemy.fourier", "gp.sky", "classical.lsq", "bayes.mcmc", "bayes.hmc", "bayes.nested"]
fig, ax = plt.subplots(figsize=(7, 1.6))
for k, n in enumerate(names):
    s = style.strategy_style(n)
    ax.plot([0, 1], [k, k], color=s.color, ls=s.linestyle, lw=3)
    ax.text(1.05, k, n, va="center")
ax.axis("off"); plt.show()
print("ffmpeg available:", writers.ffmpeg_available())

ffmpeg available: True
